In [1]:
import pandas as pd
import numpy as np

In [2]:
df=pd.read_csv(r"C:\Users\samru\OneDrive\Desktop\SIH_2025\AI-Pollution-Forecast-and-Policy-Dashboard\ML\Raw\CCR_DATA_2023_25\Raw_data_1Day_2024_site_1429_Nehru_Nagar_Delhi_DPCC_1Day.csv")

In [3]:
df

,Timestamp,PM2.5 (µg/m³),PM10 (µg/m³),NO (µg/m³),NO2 (µg/m³),NOx (ppb),NH3 (µg/m³),SO2 (µg/m³),CO (mg/m³),Ozone (µg/m³),...,MP-Xylene (µg/m³),AT (°C),RH (%),WS (m/s),WD (deg),RF (mm),TOT-RF (mm),SR (W/mt2),BP (mmHg),VWS (m/s)
0,2024-01-01,216.65,307.91,14.68,63.08,45.52,68.69,8.91,1.20,15.11,...,NaN,10.17,70.72,0.27,251.29,0.00,0.00,46.66,986.52,NaN
1,2024-01-02,243.19,349.32,29.06,68.45,60.09,64.77,13.25,1.30,21.12,...,NaN,9.25,68.98,0.26,239.07,0.00,0.00,61.47,985.79,NaN
2,2024-01-03,245.53,340.89,32.45,65.35,61.22,70.33,9.48,1.31,11.05,...,NaN,8.35,80.68,0.29,192.49,0.00,0.00,38.07,985.64,NaN
3,2024-01-04,285.09,403.48,31.29,61.74,58.31,79.99,9.81,1.21,4.05,...,NaN,9.19,78.52,0.31,297.90,0.00,0.00,19.90,985.81,NaN
4,2024-01-05,222.67,333.15,30.96,52.05,52.89,108.26,7.98,1.24,5.79,...,NaN,10.19,82.56,0.30,248.09,0.00,0.00,11.86,985.95,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
361,2024-12-27,206.84,266.27,65.07,43.20,75.87,82.05,5.40,1.18,16.99,...,NaN,13.94,91.03,1.18,170.83,0.28,0.28,4.98,993.28,NaN
362,2024-12-28,117.21,160.38,28.03,37.65,42.81,48.85,9.55,1.66,7.23,...,NaN,14.38,92.23,0.42,97.15,0.02,0.02,27.75,992.80,NaN
363,2024-12-29,123.58,171.33,8.72,23.06,19.36,50.21,7.99,1.49,23.10,...,NaN,14.09,85.18,0.30,171.33,0.00,0.00,76.91,994.40,NaN
364,2024-12-30,139.67,178.04,8.08,26.18,20.49,45.69,8.67,1.59,33.01,...,NaN,11.89,82.25,0.30,281.50,0.00,0.00,66.25,993.81,NaN


In [4]:
# ---------- 2. Remove duplicate rows and columns ----------
df = df.drop_duplicates().reset_index(drop=True)
df = df.loc[:, ~df.T.duplicated()]
print("Shape after removing duplicates:", df.shape)

Shape after removing duplicates: (366, 21)


In [5]:
# ---------- 3. Handle missing values (drop >70% NaN, impute median otherwise) ----------
nan_thresh = 0.7

# Drop columns with >70% missing
cols_to_drop = df.columns[df.isnull().mean() > nan_thresh]
df = df.drop(columns=cols_to_drop)
print(f"Dropped columns (>{int(nan_thresh*100)}% NaN): {cols_to_drop.tolist()}")

# Drop rows with >70% missing
rows_to_drop = df.index[df.isnull().mean(axis=1) > nan_thresh]
df = df.drop(index=rows_to_drop).reset_index(drop=True)
print(f"Dropped rows (>{int(nan_thresh*100)}% NaN):", len(rows_to_drop))

# Impute remaining missing values
num_cols = df.select_dtypes(include=[np.number]).columns
cat_cols = df.select_dtypes(exclude=[np.number]).columns

for col in num_cols:
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)

for col in cat_cols:
    if df[col].isnull().any():
        mode_val = df[col].mode(dropna=True)
        if not mode_val.empty:
            df[col] = df[col].fillna(mode_val[0])

print("Missing values after imputation:\n", df.isnull().sum())

Dropped columns (>70% NaN): ['Xylene (µg/m³)']
Dropped rows (>70% NaN): 0
Missing values after imputation:
 Timestamp          0
PM2.5 (µg/m³)      0
PM10 (µg/m³)       0
NO (µg/m³)         0
NO2 (µg/m³)        0
NOx (ppb)          0
NH3 (µg/m³)        0
SO2 (µg/m³)        0
CO (mg/m³)         0
Ozone (µg/m³)      0
Benzene (µg/m³)    0
Toluene (µg/m³)    0
AT (°C)            0
RH (%)             0
WS (m/s)           0
WD (deg)           0
RF (mm)            0
TOT-RF (mm)        0
SR (W/mt2)         0
BP (mmHg)          0
dtype: int64


In [6]:
# ---------- 4. Handle outliers using IQR (with 70% rule) ----------

def get_outlier_mask(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return (series < lower) | (series > upper)

# Handle columns: drop if >70% outliers, else replace with median
outlier_thresh = 0.7
cols_to_drop = []
for col in num_cols:
    outlier_mask = get_outlier_mask(df[col])
    outlier_fraction = outlier_mask.mean()
    if outlier_fraction > outlier_thresh:
        cols_to_drop.append(col)
    else:
        median_val = df[col].median()
        df.loc[outlier_mask, col] = median_val
if cols_to_drop:
    df = df.drop(columns=cols_to_drop)
    print(f"Dropped numeric columns (>{int(outlier_thresh*100)}% outliers): {cols_to_drop}")
    
# Handle rows: drop if >70% numeric columns are outliers in a given row
outlier_matrix = df[num_cols].apply(get_outlier_mask)
row_outlier_fraction = outlier_matrix.mean(axis=1)
rows_to_drop = df.index[row_outlier_fraction > outlier_thresh]
df = df.drop(index=rows_to_drop).reset_index(drop=True)
if len(rows_to_drop) > 0:
    print(f"Dropped rows (>{int(outlier_thresh*100)}% outliers): {len(rows_to_drop)}")


In [7]:
# ---------- 6. Final check ----------
print("Final shape:", df.shape)
print(df.head())

Final shape: (366, 20)
    Timestamp  PM2.5 (µg/m³)  PM10 (µg/m³)  NO (µg/m³)  NO2 (µg/m³)  \
0  2024-01-01         216.65        307.91       14.68        63.08   
1  2024-01-02         243.19        349.32       29.06        68.45   
2  2024-01-03         245.53        340.89       32.45        65.35   
3  2024-01-04         285.09        403.48       31.29        61.74   
4  2024-01-05         222.67        333.15       30.96        52.05   

   NOx (ppb)  NH3 (µg/m³)  SO2 (µg/m³)  CO (mg/m³)  Ozone (µg/m³)  \
0      45.52        68.69         8.91        1.20          15.11   
1      60.09        64.77        13.25        1.30          21.12   
2      61.22        70.33         9.48        1.31          11.05   
3      58.31        79.99         9.81        1.21           4.05   
4      52.89       108.26         7.98        1.24           5.79   

   Benzene (µg/m³)  Toluene (µg/m³)  AT (°C)  RH (%)  WS (m/s)  WD (deg)  \
0             0.37             1.63    10.17   70.72      0

In [8]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
numerics = df.select_dtypes(include=[np.number]).columns
df[numerics] = scaler.fit_transform(df[numerics])

In [9]:
df

,Timestamp,PM2.5 (µg/m³),PM10 (µg/m³),NO (µg/m³),NO2 (µg/m³),NOx (ppb),NH3 (µg/m³),SO2 (µg/m³),CO (mg/m³),Ozone (µg/m³),Benzene (µg/m³),Toluene (µg/m³),AT (°C),RH (%),WS (m/s),WD (deg),RF (mm),TOT-RF (mm),SR (W/mt2),BP (mmHg)
0,2024-01-01,1.386803,0.881231,-0.331064,2.166279,0.627236,1.091042,0.253736,-0.704297,-1.743362,1.159153,0.287206,-1.789416,0.604838,-1.092093,0.414945,0.0,0.0,-1.544137,0.858258
1,2024-01-02,1.716207,1.241426,0.526798,2.506324,1.308032,0.938214,1.698418,-0.445047,-1.419005,1.159153,0.269005,-1.911480,0.491374,-1.153795,0.231695,0.0,0.0,-1.216327,0.733560
2,2024-01-03,1.745250,1.168100,0.729034,2.310022,1.360833,1.154980,0.443475,-0.419122,-1.962478,1.885484,0.232604,-2.030891,1.254319,-0.968689,-0.466816,0.0,0.0,-1.734271,0.707938
3,2024-01-04,2.236252,1.712524,0.659832,2.081426,1.224860,1.531592,0.553324,-0.678372,-2.340265,1.958118,1.033428,-1.919441,1.113467,-0.845285,1.113906,0.0,0.0,-2.136452,0.736977
4,2024-01-05,1.461521,1.100775,0.640145,1.467825,0.971606,2.633746,-0.055839,-0.600597,-2.246358,1.377053,0.669417,-1.786762,1.376911,-0.906987,0.366958,0.0,0.0,-2.314412,0.760892
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
361,2024-12-27,1.265046,0.519035,2.675032,0.907415,2.045367,1.611904,-0.914659,-0.756148,-1.641899,0.214923,0.196203,-1.289218,1.929231,-0.289966,-0.791628,0.0,0.0,-2.466696,2.012995
362,2024-12-28,0.152595,-0.402024,0.465352,0.555972,0.500608,0.317545,0.466777,0.488255,-2.168642,-0.148243,-0.040404,-1.230840,2.007482,-0.166562,-1.896529,0.0,0.0,-1.962697,1.931002
363,2024-12-29,0.231657,-0.306778,-0.686617,-0.367913,-0.595115,0.370567,-0.052510,0.047529,-1.312145,-0.511409,-0.804827,-1.269316,1.547759,-0.906987,-0.784130,0.0,0.0,-0.874573,2.204313
364,2024-12-30,0.431360,-0.248413,-0.724798,-0.170344,-0.542314,0.194347,0.173846,0.306780,-0.777307,-0.366142,-0.713825,-1.561209,1.356697,-0.906987,0.867973,0.0,0.0,-1.110525,2.103529


In [10]:
df.to_excel("nehrunagar2024.xlsx",index=False)